# Pantanal harmonization: 50/50 FRP ensemble

Harmonize **MODIS Aqua + Terra** against **VIIRS S-NPP**, February 2012–January 2026.
This standalone notebook preserves the original input discovery, quality filters, local dates,
2 km grid, observed counts, seasonal folders and ZIP workflow. The separate MODIS backfill
notebook remains unchanged.

The calibrated estimate averages the original weekly-fitted multiplier with a monthly-fitted
FRP-band model. Expected retrospective monthly weighted absolute errors on the original inputs:

| Months | Original multiplier | FRP only | 50/50 ensemble |
|---|---:|---:|---:|
| Later 96 | 12.75% | 11.20% | 10.79% |
| 120 outside fitting | 11.51% | 10.54% | 9.94% |
| All 168, including fitting | 11.41% | 10.26% | 9.88% |

These historical periods have been inspected during model development. The figures are
exploratory retrospective comparisons, not fresh independent validation or ground-truth accuracy.
The 168-month score includes 48 fitting months. Observation coverage remains unverified.

## 1. Configure paths

**On Kaggle:** import this notebook, attach the same Pantanal dataset using **Add Input**, leave
`INPUT_PATH = None`, `TRANSFER_DIR = None`, and `PILOT_ONLY = False`, then choose **Run All**.
No GPU is needed. The notebook is self-contained and does not import project Python files.

The original input instructions and supported layouts are unchanged: paired season CSVs,
the original pilot ZIP plus later paired ZIPs, or an exact extraction of the pilot ZIP.
If multiple inputs are found, set `INPUT_PATH` to the intended dataset folder or pilot archive;
set `TRANSFER_DIR` if the later ZIPs are in another directory. Section 3 provides the same
missing-file and ambiguous-input diagnostics as the original notebook.

Outputs go to `/kaggle/working/pantanal_harmonized_frp_ensemble` on Kaggle. Locally, run from
the project root to discover `NEWDATA`; outputs go to that same named subfolder of the current
directory. Requires Python 3.11+, numpy, pandas, scipy, pyproj, and IANA timezone data (tzdata).
These are commonly available on Kaggle; if an import fails, install the missing package before
running. There are no automatic installs or internet-dependent data downloads.

In [1]:
import hashlib
import json
import numpy as np
from scipy.optimize import linprog
import io
import tempfile
from contextlib import ExitStack
from dataclasses import dataclass
from fractions import Fraction
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile
from zoneinfo import ZoneInfo

import pandas as pd
from pyproj import Transformer


# Optional input overrides.
INPUT_PATH = None
TRANSFER_DIR = None
PILOT_ONLY = False
KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
OUTPUT_DIR = (KAGGLE_WORKING if KAGGLE_WORKING.is_dir() else Path.cwd()) / "pantanal_harmonized_frp_ensemble"

## 2. Set the analysis scope

Each season runs February–January. Fit the original multiplier on full weeks in the four
seasons starting 2013–2016. Fit the three FRP coefficients on the 48 months in those same
seasons. Keep the blend fixed at 50/50 and evaluate all other seasons without refitting.
Both Aqua and Terra are included. FRP describes radiative intensity, not measured fire area.

In [2]:
PILOT_YEARS = range(2012, 2018)
TRANSFER_YEARS = range(2018, 2026)
FIT_YEARS = range(2013, 2017)
SEASONAL_YEARS = range(2012, 2017)
EARLY_YEAR, LATE_YEAR = PILOT_YEARS[0], PILOT_YEARS[-1]
TIMEZONE = "America/Cuiaba"
BOUNDS = (-58, -19.5, -56, -17)
GRID_CRS, CELL_SIZE_M = "EPSG:6933", 2000
METHOD = "pantanal_cell_days_frp_ensemble_v2"
INPUT_FIELDS = {
    "latitude", "longitude", "brightness", "scan", "track", "acq_date",
    "acq_time", "satellite", "instrument", "confidence", "version",
    "bright_t31", "frp", "daynight", "type",
}
PRODUCTS = {
    "MODIS": ({"Aqua", "Terra"}, "MODIS", {"6.03", "61.03"}),
    "VIIRS": ({"SNPP"}, "SNPP", {"2"}),
}
PLATFORMS = {"Aqua": "aqua_cell_days", "Terra": "terra_cell_days", "SNPP": "snpp_cell_days"}
MODIS = "modis_cell_days"
SNPP = PLATFORMS["SNPP"]
ESTIMATE = "calibrated_modis_cell_days"
PERIOD_KEYS = ["period_kind", "period_start_local"]
SEASONAL_METHODS = {
    "modis_high_on_snpp_p75": "Unadjusted MODIS on S-NPP threshold",
    "modis_high_on_own_p75": "Unadjusted MODIS on its own threshold",
    "calibrated_modis_high_on_snpp_p75": "Calibrated MODIS on S-NPP threshold",
}


@dataclass
class Period:
    calendar: pd.DatetimeIndex
    detections: pd.DataFrame
    provenance: dict[str, str]


def folder(year):
    return f"{year}-{year + 1}"

## 3. Locate the raw source data

The notebook recognizes paired season CSVs, the original pilot ZIP, or its exact extraction. It checks the configured path or searches Kaggle's input mount. Set INPUT_PATH to the intended folder or archive if multiple copies are attached.

In [3]:
ORIGINAL_ARCHIVE_SHA256 = "0f3714265302cf0b6d4d3909376b0d45aee16df8ae2217418cbf856888802b03"
ORIGINAL_MEMBERS = {'2012-2013/fire_archive_MODIS_845351.csv': 'a2eb11e41fafce4e8454d9d9fcf2e58143e1560e66a5e8f89fc86bba4d960517',
 '2012-2013/fire_archive_VIIRS_845352.csv': 'eeae552892ea655dfe3d3a5cd58cb33d2a8fb616869fa9cb1b6a5ead043fc9a9',
 '2013-2014/fire_archive_MODIS_845355.csv': 'eca5b6df8f8197112727ba4f9d0bfb900250af19ae813ec185e47da2191a5762',
 '2013-2014/fire_archive_VIIRS_845356.csv': '5867169e02fc976b35ed2e02200f6e81288c192b5f30add8550c1b23a5833f9a',
 '2014-2015/fire_archive_MODIS_845359.csv': '2215804c488377ed6dc5173e77e597eb6d5bb3e2e87d8847d559aa0444cd142c',
 '2014-2015/fire_archive_VIIRS_845360.csv': '062a1576bc648318ce09ffd328885f8d652444453e11663449fc251166aa400e',
 '2015-2016/fire_archive_MODIS_845361.csv': 'e1933b59c42d8d37218da8e1a3b24b2f3992b60c61e9545064a346782386f329',
 '2015-2016/fire_archive_VIIRS_845362.csv': 'df9612171a95f39843006d85ec75a9e47d45db22539dde91a6a198eb296af7fd',
 '2016-2017/fire_archive_MODIS_845363.csv': '560cf02a193f353c5bf62059e03b85b5878da566097f1ed53908553b3c134afa',
 '2016-2017/fire_archive_VIIRS_845364.csv': '54d2b10241987a7be2ac960ab93edecd84fd781022b35ea89d4aefc274ea051d',
 '2017-2018/fire_archive_MODIS_845365.csv': '7e6fee1102094fb91210658135f7a0274f90447416e357eea4c84c29cadff1d5',
 '2017-2018/fire_archive_VIIRS_845366.csv': '781c134f0314db2bc4343cdeb2b3e600d4e92f00f32539cec43666314d544ac2'}


ARCHIVE_NAME = "Patanel Data.zip"
RUN_YEARS = list(PILOT_YEARS) + ([] if PILOT_ONLY else list(TRANSFER_YEARS))


def season_csvs(directory, years):
    return {
        (year, product): directory / folder(year) / f"{product}_{folder(year)}.csv"
        for year in years
        for product in PRODUCTS
    }


def locate_input(configured_path):
    if configured_path is not None:
        root = Path(configured_path).expanduser().resolve(strict=True)
        if root.is_file():
            return root
        roots = [root]
    elif KAGGLE_INPUT.is_dir():
        roots = [KAGGLE_INPUT]
    else:
        # Prefer the local order archives when the workspace contains both layouts.
        for directory in (Path.cwd() / "NEWDATA", Path.cwd()):
            archive = directory / ARCHIVE_NAME
            if archive.is_file():
                return archive
        roots = [Path.cwd() / "NEWDATA", Path.cwd()]

    first_original = Path(next(iter(ORIGINAL_MEMBERS)))
    first_season = folder(PILOT_YEARS[0])
    # Use both sensors and all expected seasons as markers so incomplete
    # datasets still get a diagnostic listing their missing files.
    season_names = {p.name for p in season_csvs(Path(), RUN_YEARS).values()}
    archives, extracted, paired = set(), set(), set()
    for root in roots:
        for path in root.rglob("*"):
            if path.name.casefold() == ARCHIVE_NAME.casefold() and path.is_file():
                archives.add(path)
            elif path.name == first_original.name and path.parent.name == first_season:
                extracted.add(path.parent.parent)
            elif path.name in season_names and path.name.endswith(f"_{path.parent.name}.csv"):
                paired.add(path.parent.parent)

    # A complete paired tree can run even if its directory also holds a pilot ZIP.
    complete_pairs = {
        directory for directory in paired
        if all(path.is_file() for path in season_csvs(directory, RUN_YEARS).values())
    }
    sibling_archives = {p.parent for p in archives}
    matches = sorted(
        complete_pairs
        | {p for p in archives if p.parent not in complete_pairs}
        | (extracted - sibling_archives - complete_pairs)
        | (paired - complete_pairs - sibling_archives - extracted)
    )
    if len(matches) == 1:
        return matches[0]
    if matches:
        paths = ", ".join(map(str, matches))
        raise ValueError(f"Multiple Pantanal inputs found. Set INPUT_PATH to one of: {paths}")

    searched = ", ".join(map(str, roots))
    raise FileNotFoundError(
        f"No Pantanal input found under {searched}. On Kaggle, attach the dataset with Add Input. "
        "Use the folder containing 2012-2013/MODIS_2012-2013.csv and "
        "2012-2013/VIIRS_2012-2013.csv, or use the original Patanel Data.zip."
    )


input_path = locate_input(INPUT_PATH)
print(f"Input {input_path}")
print(f"Output {OUTPUT_DIR}")

paired_csvs = {}
transfer_path = None
if input_path.is_dir():
    expected_csvs = season_csvs(input_path, RUN_YEARS)
    if any(path.is_file() for path in expected_csvs.values()):
        missing = [path.relative_to(input_path).as_posix() for path in expected_csvs.values() if not path.is_file()]
        if missing:
            raise FileNotFoundError(f"Missing raw seasonal CSVs under {input_path}: {', '.join(missing)}")
        paired_csvs = expected_csvs

if not PILOT_ONLY and not paired_csvs:
    if TRANSFER_DIR is None:
        transfer_path = input_path.parent if input_path.is_file() else input_path
    else:
        transfer_path = Path(TRANSFER_DIR).expanduser().resolve(strict=True)
    missing = [
        f"{product} {folder(year)}.zip"
        for year in TRANSFER_YEARS
        for product in PRODUCTS
        if not (transfer_path / f"{product} {folder(year)}.zip").is_file()
    ]
    if missing:
        names = ", ".join(missing)
        raise FileNotFoundError(
            f"Missing later FIRMS orders in {transfer_path}: {names}. "
            "Attach the paired ZIPs or set TRANSFER_DIR to their directory."
        )
    print(f"Later source orders {transfer_path}")


Input <FIRMS archive>.zip
Output pantanal_harmonized_frp_ensemble
Later source orders <later FIRMS orders>


## 4. Read and validate the source records

Verify the CSV schema and sensor versions, clip to the Pantanal box, and convert UTC times to local dates in **America/Cuiaba**. The extraction check verifies each original CSV by its SHA-256 hash so Kaggle's unpacked input retains the same provenance as the ZIP.

In [4]:
def read_source(payload, member, product, dates, local_zone, transformer, *, pilot=False):
    data = pd.read_csv(io.BytesIO(payload), dtype=str, keep_default_na=False)
    if set(data.columns) != INPUT_FIELDS or data.empty:
        raise ValueError(f"Missing records or unexpected CSV columns in {member}")
    satellites, instrument, versions = PRODUCTS[product]
    if pilot and product == "MODIS":
        versions = {"6.03"}
    if (not data["satellite"].isin(satellites).all()
            or not data["instrument"].eq(instrument).all()
            or not data["version"].isin(versions).all()):
        raise ValueError(f"Unexpected satellite, instrument or product version in {member}")
    version = "|".join(sorted(data["version"].unique()))

    raw_count = len(data)
    data = data.drop_duplicates().copy()
    duplicates = raw_count - len(data)
    hhmm = data["acq_time"].str.zfill(4)
    valid_times = (data["acq_date"].str.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}")
                   & data["acq_time"].str.fullmatch(r"[0-9]{1,4}")
                   & hhmm.str[:2].lt("24") & hhmm.str[2:].lt("60"))
    if not valid_times.all():
        raise ValueError(f"Invalid date or HHMM acquisition time in {member}")
    data["acquired_at_utc"] = pd.to_datetime(
        data["acq_date"] + " " + hhmm, format="%Y-%m-%d %H%M", utc=True,
    )
    data["local_date"] = data["acquired_at_utc"].dt.tz_convert(local_zone).dt.tz_localize(None).dt.normalize()
    if not data["local_date"].between(dates[0], dates[-1]).all():
        raise ValueError(f"Local acquisition outside the folder's dates in {member}")

    lon = pd.to_numeric(data["longitude"], errors="coerce")
    lat = pd.to_numeric(data["latitude"], errors="coerce")
    if not (lon.between(-180, 180) & lat.between(-90, 90)).all():
        raise ValueError(f"Invalid longitude or latitude in {member}")
    if not (data["type"].isin(["0", "1", "2", "3"]) & data["daynight"].isin(["D", "N"])).all():
        raise ValueError(f"Invalid hotspot type or day/night flag in {member}")

    if product == "MODIS":
        confidence = pd.to_numeric(data["confidence"], errors="coerce")
        valid_confidence = confidence.between(0, 100) & confidence.mod(1).eq(0)
        confidence_pass = confidence.ge(30)
    else:
        valid_confidence = data["confidence"].isin(["l", "n", "h"])
        confidence_pass = data["confidence"].isin(["n", "h"])
    if not valid_confidence.all():
        raise ValueError(f"Invalid {product} confidence in {member}")

    west, south, east, north = BOUNDS
    in_area = lon.between(west, east) & lat.between(south, north)
    vegetation = in_area & data["type"].eq("0")
    keep = vegetation & confidence_pass
    selected = data.loc[keep, ["local_date", "acquired_at_utc", "satellite", "daynight"]].copy()
    selected["frp"] = pd.to_numeric(data.loc[keep, "frp"], errors="raise")
    if product == "MODIS" and (not np.isfinite(selected["frp"]).all() or selected["frp"].lt(0).any()):
        raise ValueError(f"Invalid qualifying MODIS FRP in {member}")
    east_m, north_m = transformer.transform(lon[keep].to_numpy(), lat[keep].to_numpy(), errcheck=True)
    selected["cell_x"] = (east_m // CELL_SIZE_M).astype("int64")
    selected["cell_y"] = (north_m // CELL_SIZE_M).astype("int64")
    print(f"{member} (version {version}) | raw {raw_count:,} | retained {len(selected):,} | "
          f"duplicates {duplicates:,} | outside bbox {int((~in_area).sum()):,} | "
          f"other types {int((in_area & ~vegetation).sum()):,} | "
          f"low confidence {int((vegetation & ~confidence_pass).sum()):,}")
    return selected, version


def read_pilot(members, read_member, digest):
    local_zone = ZoneInfo(TIMEZONE)
    transformer = Transformer.from_crs("EPSG:4326", GRID_CRS, always_xy=True)
    periods = {}
    for year in PILOT_YEARS:
        dates = pd.date_range(f"{year}-02-01", f"{year + 1}-02-01", inclusive="left")
        frames, provenance = [], {}
        for product in PRODUCTS:
            prefix = f"{folder(year)}/fire_archive_{product}_"
            matches = [m for m in members if m.startswith(prefix) and m.endswith(".csv")]
            if len(matches) != 1:
                raise ValueError(f"Expected exactly one {product} CSV for {folder(year)}")
            member = matches[0]
            frame, version = read_source(
                read_member(member), member, product, dates, local_zone, transformer, pilot=True,
            )
            frames.append(frame)
            provenance[f"{product.lower()}_source_member"] = member
            provenance[f"{product.lower()}_version"] = version
        periods[year] = Period(dates, pd.concat(frames, ignore_index=True), provenance)
    print(f"Pilot archive SHA-256 {digest}")
    return periods, digest


def read_archive(path):
    with path.open("rb") as source:
        digest = hashlib.file_digest(source, "sha256").hexdigest()
        source.seek(0)
        with ZipFile(source) as archive:
            return read_pilot(archive.namelist(), archive.read, digest)


def read_transfer_archives(directory):
    local_zone = ZoneInfo(TIMEZONE)
    transformer = Transformer.from_crs("EPSG:4326", GRID_CRS, always_xy=True)
    periods = {}
    for year in TRANSFER_YEARS:
        dates = pd.date_range(f"{year}-02-01", f"{year + 1}-02-01", inclusive="left")
        frames, provenance, archive_details = [], {}, {}
        for product, prefix in (("MODIS", "fire_archive_M-C61_"), ("VIIRS", "fire_archive_SV-C2_")):
            path = directory / f"{product} {folder(year)}.zip"
            with path.open("rb") as source:
                digest = hashlib.file_digest(source, "sha256").hexdigest()
                source.seek(0)
                with ZipFile(source) as archive:
                    members = [m for m in archive.namelist() if m.lower().endswith(".csv")]
                    if len(members) != 1 or not Path(members[0]).name.startswith(prefix):
                        raise ValueError(f"Expected one {product} FIRMS CSV in {path}")
                    member = members[0]
                    frame, version = read_source(
                        archive.read(member), member, product, dates, local_zone, transformer,
                    )
            frames.append(frame)
            name = product.lower()
            provenance[f"{name}_source_member"] = member
            provenance[f"{name}_version"] = version
            archive_details[f"{name}_source_archive"] = path.name
            archive_details[f"{name}_source_archive_sha256"] = digest
        periods[year] = Period(dates, pd.concat(frames, ignore_index=True), provenance | archive_details)
    return periods


def read_input(path):
    if path.is_file():
        return read_archive(path)

    files = {}
    for member, expected_digest in ORIGINAL_MEMBERS.items():
        csv_path = path / member
        payload = csv_path.read_bytes()
        if hashlib.sha256(payload).hexdigest() != expected_digest:
            raise ValueError(f"{csv_path} differs from the original archive. Use the original ZIP or its exact extraction.")
        files[member] = payload
    print(f"Verified {len(files)} extracted CSVs against the original Pantanal archive.")
    return read_pilot(files, files.__getitem__, ORIGINAL_ARCHIVE_SHA256)


def read_season_csvs(paths):
    local_zone = ZoneInfo(TIMEZONE)
    transformer = Transformer.from_crs("EPSG:4326", GRID_CRS, always_xy=True)
    periods = {}
    pilot_members = {
        (int(member[:4]), Path(member).name.split("_")[2]): (member, digest)
        for member, digest in ORIGINAL_MEMBERS.items()
    }
    for year in RUN_YEARS:
        pilot = year in PILOT_YEARS
        dates = pd.date_range(f"{year}-02-01", f"{year + 1}-02-01", inclusive="left")
        frames, provenance = [], {}
        for product in PRODUCTS:
            path = paths[year, product]
            member = f"{folder(year)}/{path.name}"
            payload = path.read_bytes()
            digest = hashlib.sha256(payload).hexdigest()
            if pilot:
                _, expected_digest = pilot_members[year, product]
                if digest != expected_digest:
                    raise ValueError(f"{path} differs from the original pilot CSV. Use the raw FIRMS export from the project.")
            frame, version = read_source(payload, member, product, dates, local_zone, transformer, pilot=pilot)
            frames.append(frame)
            name = product.lower()
            provenance[f"{name}_source_member"] = member
            provenance[f"{name}_version"] = version
            provenance[f"{name}_source_csv_sha256"] = digest
        periods[year] = Period(dates, pd.concat(frames, ignore_index=True), provenance)
    return periods, ORIGINAL_ARCHIVE_SHA256


if paired_csvs:
    periods, digest = read_season_csvs(paired_csvs)
else:
    periods, digest = read_input(input_path)
if transfer_path is not None:
    periods.update(read_transfer_archives(transfer_path))

2012-2013/fire_archive_MODIS_845351.csv (version 6.03) | raw 6,986 | retained 6,560 | duplicates 0 | outside bbox 0 | other types 5 | low confidence 421
2012-2013/fire_archive_VIIRS_845352.csv (version 2) | raw 27,831 | retained 24,873 | duplicates 0 | outside bbox 0 | other types 66 | low confidence 2,892


2013-2014/fire_archive_MODIS_845355.csv (version 6.03) | raw 3,437 | retained 3,205 | duplicates 0 | outside bbox 0 | other types 5 | low confidence 227
2013-2014/fire_archive_VIIRS_845356.csv (version 2) | raw 12,490 | retained 11,571 | duplicates 0 | outside bbox 0 | other types 19 | low confidence 900
2014-2015/fire_archive_MODIS_845359.csv (version 6.03) | raw 1,024 | retained 973 | duplicates 0 | outside bbox 0 | other types 2 | low confidence 49
2014-2015/fire_archive_VIIRS_845360.csv (version 2) | raw 4,257 | retained 3,749 | duplicates 0 | outside bbox 0 | other types 0 | low confidence 508
2015-2016/fire_archive_MODIS_845361.csv (version 6.03) | raw 2,595 | retained 2,474 | duplicates 0 | outside bbox 0 | other types 7 | low confidence 114
2015-2016/fire_archive_VIIRS_845362.csv (version 2) | raw 8,382 | retained 7,431 | duplicates 0 | outside bbox 0 | other types 4 | low confidence 947


2016-2017/fire_archive_MODIS_845363.csv (version 6.03) | raw 3,415 | retained 3,217 | duplicates 0 | outside bbox 0 | other types 7 | low confidence 191
2016-2017/fire_archive_VIIRS_845364.csv (version 2) | raw 12,948 | retained 11,203 | duplicates 0 | outside bbox 0 | other types 22 | low confidence 1,723
2017-2018/fire_archive_MODIS_845365.csv (version 6.03) | raw 3,300 | retained 3,163 | duplicates 0 | outside bbox 0 | other types 5 | low confidence 132
2017-2018/fire_archive_VIIRS_845366.csv (version 2) | raw 12,502 | retained 10,778 | duplicates 0 | outside bbox 0 | other types 19 | low confidence 1,705
Pilot archive SHA-256 0f3714265302cf0b6d4d3909376b0d45aee16df8ae2217418cbf856888802b03


fire_archive_M-C61_848145.csv (version 6.03) | raw 1,525 | retained 1,414 | duplicates 0 | outside bbox 0 | other types 7 | low confidence 104
fire_archive_SV-C2_848146.csv (version 2) | raw 5,595 | retained 4,971 | duplicates 0 | outside bbox 0 | other types 32 | low confidence 592
fire_archive_M-C61_848152.csv (version 6.03) | raw 9,360 | retained 8,670 | duplicates 0 | outside bbox 0 | other types 3 | low confidence 687


fire_archive_SV-C2_848153.csv (version 2) | raw 35,698 | retained 31,800 | duplicates 0 | outside bbox 0 | other types 124 | low confidence 3,774
fire_archive_M-C61_848157.csv (version 6.03) | raw 26,054 | retained 23,622 | duplicates 0 | outside bbox 0 | other types 15 | low confidence 2,417


fire_archive_SV-C2_848158.csv (version 2) | raw 110,819 | retained 101,182 | duplicates 0 | outside bbox 0 | other types 1,149 | low confidence 8,488
fire_archive_M-C61_848159.csv (version 6.03) | raw 8,166 | retained 7,467 | duplicates 0 | outside bbox 0 | other types 9 | low confidence 690


fire_archive_SV-C2_848160.csv (version 2) | raw 32,482 | retained 30,019 | duplicates 0 | outside bbox 0 | other types 378 | low confidence 2,085
fire_archive_M-C61_848163.csv (version 6.03|61.03) | raw 1,449 | retained 1,278 | duplicates 0 | outside bbox 0 | other types 1 | low confidence 170
fire_archive_SV-C2_848164.csv (version 2) | raw 6,168 | retained 5,577 | duplicates 0 | outside bbox 0 | other types 166 | low confidence 425
fire_archive_M-C61_848165.csv (version 61.03) | raw 8,977 | retained 8,312 | duplicates 0 | outside bbox 0 | other types 11 | low confidence 654


fire_archive_SV-C2_848166.csv (version 2) | raw 35,660 | retained 32,975 | duplicates 0 | outside bbox 0 | other types 302 | low confidence 2,383
fire_archive_M-C61_848171.csv (version 61.03) | raw 17,259 | retained 15,485 | duplicates 0 | outside bbox 0 | other types 8 | low confidence 1,766


fire_archive_SV-C2_848172.csv (version 2) | raw 78,634 | retained 73,001 | duplicates 0 | outside bbox 0 | other types 1,448 | low confidence 4,185
fire_archive_M-C61_848173.csv (version 61.03) | raw 648 | retained 601 | duplicates 0 | outside bbox 0 | other types 4 | low confidence 43
fire_archive_SV-C2_848174.csv (version 2) | raw 2,880 | retained 2,631 | duplicates 0 | outside bbox 0 | other types 44 | low confidence 205


## 5. Count hotspot cell-days

Count a cell once per local day for each satellite. The combined MODIS count is the union of Terra and Aqua cell-days. Build week and month totals independently from the daily counts. Mark weeks clipped by a year-folder boundary as partial.

In [5]:
def daily_counts(period, daynight=None):
    data = period.detections
    if daynight is not None:
        data = data.loc[data["daynight"].eq(daynight)]
    cell_day = ["local_date", "cell_x", "cell_y"]
    platform_cells = data.drop_duplicates([*cell_day, "satellite"])
    by_platform = platform_cells.groupby(["local_date", "satellite"]).size().unstack(fill_value=0)
    daily = by_platform.reindex(index=period.calendar, columns=list(PLATFORMS), fill_value=0).rename(columns=PLATFORMS)
    modis_cells = platform_cells.loc[platform_cells["satellite"].ne("SNPP"), cell_day].drop_duplicates()
    daily[MODIS] = modis_cells.groupby("local_date").size().reindex(period.calendar, fill_value=0)
    return daily


def aggregate(daily, grain):
    dates = daily.index
    if grain == "week":
        keys = dates - pd.to_timedelta(dates.day_of_week, unit="D")
    else:
        keys = dates.to_period("M").to_timestamp()
    totals = daily.groupby(keys).sum()
    bounds = dates.to_series().groupby(keys).agg(["min", "max", "size"])
    totals.insert(0, "period_kind", grain)
    totals.insert(1, "period_start_local", bounds["min"])
    totals.insert(2, "period_end_local", bounds["max"])
    totals.insert(3, "is_partial_week", (grain == "week") & bounds["size"].ne(7))
    return totals.reset_index(drop=True)


def validate_rollups(year, daily, weekly, monthly):
    if len(weekly) != 53 or len(monthly) != 12:
        raise ValueError(f"Unexpected weekly or monthly row count for {folder(year)}")
    for result in (weekly, monthly):
        first = result["period_start_local"]
        last = result["period_end_local"]
        if (first.iloc[0] != daily.index[0] or last.iloc[-1] != daily.index[-1]
                or not first.iloc[1:].reset_index(drop=True).eq(last.iloc[:-1].reset_index(drop=True) + pd.Timedelta(days=1)).all()
                or not daily.sum().eq(result[daily.columns].sum()).all()):
            raise ValueError(f"Period boundaries or observed counts failed to reconcile for {folder(year)}")
    modis_platforms = daily[[PLATFORMS["Aqua"], PLATFORMS["Terra"]]]
    if not (daily[MODIS].ge(modis_platforms.max(axis=1)) & daily[MODIS].le(modis_platforms.sum(axis=1))).all():
        raise ValueError(f"MODIS union counts failed to reconcile for {folder(year)}")


weekly, monthly, full_weeks = {}, {}, {}
for year, period in periods.items():
    daily = daily_counts(period)
    weekly[year] = aggregate(daily, "week")
    monthly[year] = aggregate(daily, "month")
    validate_rollups(year, daily, weekly[year], monthly[year])
    weeks = weekly[year]
    full_weeks[year] = weeks.loc[~weeks["is_partial_week"]]
    print(f"{folder(year)} | {len(weeks)} weekly rows | {len(monthly[year])} monthly rows")

2012-2013 | 53 weekly rows | 12 monthly rows
2013-2014 | 53 weekly rows | 12 monthly rows
2014-2015 | 53 weekly rows | 12 monthly rows
2015-2016 | 53 weekly rows | 12 monthly rows
2016-2017 | 53 weekly rows | 12 monthly rows
2017-2018 | 53 weekly rows | 12 monthly rows
2018-2019 | 53 weekly rows | 12 monthly rows
2019-2020 | 53 weekly rows | 12 monthly rows
2020-2021 | 53 weekly rows | 12 monthly rows


2021-2022 | 53 weekly rows | 12 monthly rows
2022-2023 | 53 weekly rows | 12 monthly rows
2023-2024 | 53 weekly rows | 12 monthly rows
2024-2025 | 53 weekly rows | 12 monthly rows
2025-2026 | 53 weekly rows | 12 monthly rows


## 6. Fit the original multiplier and the FRP-band model

1. Fit the original multiplier by minimizing weekly absolute error (expected **1.84**).
2. For each combined MODIS cell-day, take the maximum FRP across qualifying Aqua/Terra
   detections. Count that cell-day exactly once, in one band: **below 10 MW**, **10–below
   50 MW**, or **at least 50 MW**. Sum the bands independently into weeks and months.
3. Fit three nonnegative coefficients (bounded 0–6) to minimize monthly absolute error in
   the same fitting seasons. A linear-programming solver finds the coefficients.
4. Average the original estimate and the FRP estimate with fixed equal weights.

Expected fitted FRP coefficients are approximately **1.14224, 2.42451, 0.95231**. They are
recomputed from the training data below; evaluation targets never enter the fit.
The names `small_fire_cell_days` / `large_fire_cell_days` refer to FRP bands, not fire area.

In [6]:
def fit_multiplier(weeks):
    # For positive MODIS count m, |k*m-v| = m*|k-v/m|. Zero counts add a constant.
    ratios = sorted((Fraction(int(v), int(m)), int(m)) for m, v in zip(weeks[MODIS], weeks[SNPP]) if m > 0)
    remaining_weight = sum(weight for _, weight in ratios)
    for ratio, weight in ratios:
        remaining_weight -= 2 * weight
        if remaining_weight <= 0:
            return ratio
    raise ValueError("The calibration weeks contain no positive MODIS counts")


def fit_for_years(full_weeks, years):
    return fit_multiplier(pd.concat([full_weeks[year] for year in years], ignore_index=True))


multiplier = fit_for_years(full_weeks, FIT_YEARS)
training_weeks = sum(len(full_weeks[year]) for year in FIT_YEARS)
print(f"Selected factor {float(multiplier):.8f} fitted on {training_weeks} full weeks")

FRP_COLUMNS = ["small_fire_cell_days", "medium_frp_cell_days", "large_fire_cell_days"]

def add_frp_features(period, rows):
    modis = period.detections.loc[period.detections["satellite"].isin(["Aqua", "Terra"])]
    cells = modis.groupby(["local_date", "cell_x", "cell_y"])["frp"].max().reset_index()
    daily = pd.DataFrame(index=period.calendar)
    bands = [cells.frp.lt(10), cells.frp.ge(10) & cells.frp.lt(50), cells.frp.ge(50)]
    for column, mask in zip(FRP_COLUMNS, bands):
        daily[column] = cells.loc[mask].groupby("local_date").size().reindex(period.calendar, fill_value=0)
    result = rows.copy()
    for column in FRP_COLUMNS:
        result[column] = [int(daily.loc[start:end, column].sum()) for start, end in
                          zip(rows.period_start_local, rows.period_end_local)]
    if not result[FRP_COLUMNS].sum(axis=1).eq(result[MODIS]).all():
        raise ValueError("FRP bands do not partition the original MODIS cell-days")
    return result

for year, period in periods.items():
    weekly[year] = add_frp_features(period, weekly[year])
    monthly[year] = add_frp_features(period, monthly[year])
    if not weekly[year][FRP_COLUMNS].sum().eq(monthly[year][FRP_COLUMNS].sum()).all():
        raise ValueError("Weekly/monthly FRP-band totals do not reconcile")
    full_weeks[year] = weekly[year].loc[~weekly[year].is_partial_week]

training_months = pd.concat([monthly[y] for y in FIT_YEARS], ignore_index=True)
X = training_months[FRP_COLUMNS].to_numpy(dtype=float)
y = training_months[SNPP].to_numpy(dtype=float)
n = len(y)
# Auxiliary residual variables turn sum(abs(X @ coefficients - y)) into a linear program.
solution = linprog(np.r_[np.zeros(3), np.ones(n)],
                  A_ub=np.vstack([np.c_[X, -np.eye(n)], np.c_[-X, -np.eye(n)]]),
                  b_ub=np.r_[y, -y], bounds=[(0, 6)] * 3 + [(0, None)] * n, method="highs")
if not solution.success:
    raise ValueError(f"FRP calibration failed: {solution.message}")
frp_coefficients = solution.x[:3]

def predict(rows, method="ensemble"):
    baseline = rows[MODIS].to_numpy(dtype=float) * float(multiplier)
    if method == "baseline":
        return baseline
    frp_estimate = rows[FRP_COLUMNS].to_numpy(dtype=float) @ frp_coefficients
    if method == "frp_only":
        return frp_estimate
    if method != "ensemble":
        raise ValueError(f"Unknown method: {method}")
    return 0.5 * baseline + 0.5 * frp_estimate

print(f"FRP coefficients fitted on {n} training months: {frp_coefficients}")
print("Final estimate = 0.5 * original estimate + 0.5 * FRP estimate")

Selected factor 1.84000000 fitted on 206 full weeks


FRP coefficients fitted on 48 training months: [1.14223748 2.4245066  0.95230674]
Final estimate = 0.5 * original estimate + 0.5 * FRP estimate


## 7. Compare all three techniques

WAE = `100 × sum(abs(predicted − S-NPP)) / sum(S-NPP)`. This pools absolute errors,
not individual monthly percentages. Score every eligible period, including zero MODIS counts
with prediction zero. Exclude only partial weeks from weekly evaluation. Also show cell-day
MAE and signed bias. All-month results include training data.

With `PILOT_ONLY = True`, the later-month group is omitted and group sizes reflect only the
available pilot seasons. A full run produces the **96, 120, and 168 month** comparisons.

In [7]:
def score(rows, method):
    residual = predict(rows, method) - rows[SNPP].to_numpy(dtype=float)
    total = float(rows[SNPP].sum())
    return {"periods": len(rows),
            "weighted_absolute_error_pct": float(100 * np.abs(residual).sum() / total) if total else None,
            "mae_cell_days": float(np.abs(residual).mean()),
            "bias_pct": float(100 * residual.sum() / total) if total else None}

evaluation_groups = {
    "later_orders": [year for year in periods if year in TRANSFER_YEARS],
    "outside_fitting": [year for year in periods if year not in FIT_YEARS],
    "all_including_fitting": list(periods),
}
comparisons, season_comparisons = [], []
for grain, tables in (("month", monthly), ("full_week", full_weeks)):
    for group, years in evaluation_groups.items():
        if not years:
            continue
        rows = pd.concat([tables[year] for year in years], ignore_index=True)
        for method in ("baseline", "frp_only", "ensemble"):
            comparisons.append({"group": group, "grain": grain, "method": method, **score(rows, method)})
    for year, rows in tables.items():
        for method in ("baseline", "frp_only", "ensemble"):
            season_comparisons.append({"season": folder(year), "grain": grain,
                                       "method": method, **score(rows, method)})
comparison = pd.DataFrame(comparisons)
per_season_comparison = pd.DataFrame(season_comparisons)
print(comparison.to_string(index=False, float_format=lambda value: f"{value:.6f}"))

                group     grain   method  periods  weighted_absolute_error_pct  mae_cell_days  bias_pct
         later_orders     month baseline       96                    12.752899      87.995000  2.744444
         later_orders     month frp_only       96                    11.204512      77.311135 -5.480755
         later_orders     month ensemble       96                    10.786363      74.425907 -1.368155
      outside_fitting     month baseline      120                    11.508329      75.135000  2.182296
      outside_fitting     month frp_only      120                    10.535852      68.785946 -5.032255
      outside_fitting     month ensemble      120                     9.940455      64.898745 -1.424980
all_including_fitting     month baseline      168                    11.408430      61.978333  1.508552
all_including_fitting     month frp_only      168                    10.263806      55.759957 -4.339704
all_including_fitting     month ensemble      168               

## 8. Check unusually active months

Reproduce the original 2017–2018 seasonal diagnostic using earlier same-month values. It remains an evaluation for that season.

The calibrated high-activity flag uses the new ensemble estimate; observed thresholds and flags remain unchanged.

In [8]:
def evaluate_seasons(months, reference_years, multiplier):
    history = pd.concat([months[year] for year in reference_years], ignore_index=True)
    history = history.loc[history[MODIS].gt(0) & history[SNPP].gt(0)]
    grouped = history.groupby(history["period_start_local"].dt.month)[[MODIS, SNPP]]
    reference = pd.concat([
        grouped.size().rename("seasonal_reference_year_count"),
        grouped.median().rename(columns={MODIS: "modis_reference_median_cell_days", SNPP: "snpp_reference_median_cell_days"}),
        grouped.quantile(0.75, interpolation="linear").rename(columns={MODIS: "modis_reference_p75_cell_days", SNPP: "snpp_reference_p75_cell_days"}),
    ], axis=1)
    latest = months[LATE_YEAR].copy()
    latest = latest.join(reference, on=latest["period_start_local"].dt.month)
    latest["seasonal_reference_year_count"] = latest["seasonal_reference_year_count"].fillna(0).astype("int64")
    eligible = latest["seasonal_reference_year_count"].ge(3) & latest[MODIS].gt(0) & latest[SNPP].gt(0)
    snpp_p75, modis_p75 = latest["snpp_reference_p75_cell_days"], latest["modis_reference_p75_cell_days"]
    latest = latest.assign(
        seasonal_eligible=eligible,
        snpp_high=latest[SNPP].gt(snpp_p75).astype("boolean").where(eligible),
        modis_high_on_snpp_p75=latest[MODIS].gt(snpp_p75).astype("boolean").where(eligible),
        modis_high_on_own_p75=latest[MODIS].gt(modis_p75).astype("boolean").where(eligible),
        calibrated_modis_high_on_snpp_p75=pd.Series(predict(latest), index=latest.index).gt(snpp_p75).astype("boolean").where(eligible),
    )
    fields = [*reference.columns, "seasonal_eligible", "snpp_high", *SEASONAL_METHODS]
    return latest[[*PERIOD_KEYS, *fields]]


def print_seasonal_validation(seasons, label):
    eligible = seasons.loc[seasons["seasonal_eligible"]]
    observed = eligible["snpp_high"]
    dates = eligible["period_start_local"].dt.strftime("%Y-%m")
    high_count = int(observed.sum())
    print(f"\nSeasonal classification with {label} | eligible {len(eligible)}/{len(seasons)} months")
    print(f"S-NPP high months {', '.join(dates[observed]) or 'none'}")
    for column, method in SEASONAL_METHODS.items():
        classified = eligible[column]
        retained = f"{int((classified & observed).sum())}/{high_count}" if high_count else "unavailable"
        misses = ", ".join(dates[observed & ~classified]) or "none"
        extras = ", ".join(dates[classified & ~observed]) or "none"
        print(f"{method} | disagreements {int(classified.ne(observed).sum())}/{len(eligible)} | "
              f"retained {retained} | missed {misses} | additional {extras}")


seasons = evaluate_seasons(monthly, SEASONAL_YEARS, multiplier)
print_seasonal_validation(seasons, "the first five folders")
sensitivity = evaluate_seasons(monthly, FIT_YEARS, multiplier)
print_seasonal_validation(sensitivity, "the earliest folder removed")


Seasonal classification with the first five folders | eligible 12/12 months
S-NPP high months 2017-09
Unadjusted MODIS on S-NPP threshold | disagreements 1/12 | retained 0/1 | missed 2017-09 | additional none
Unadjusted MODIS on its own threshold | disagreements 2/12 | retained 1/1 | missed none | additional 2017-05, 2017-07
Calibrated MODIS on S-NPP threshold | disagreements 2/12 | retained 1/1 | missed none | additional 2017-05, 2017-07

Seasonal classification with the earliest folder removed | eligible 12/12 months
S-NPP high months 2017-04, 2017-05, 2017-07, 2017-09
Unadjusted MODIS on S-NPP threshold | disagreements 3/12 | retained 1/4 | missed 2017-04, 2017-05, 2017-09 | additional none
Unadjusted MODIS on its own threshold | disagreements 2/12 | retained 4/4 | missed none | additional 2017-02, 2017-03
Calibrated MODIS on S-NPP threshold | disagreements 1/12 | retained 3/4 | missed 2017-04 | additional none


## 9. Export activity and reproducibility information

Each seasonal CSV retains the original columns and 53 weekly + 12 monthly rows. Five extra
columns record low/high FRP counts, the original estimate, the baseline multiplier, and model
status. `calibrated_modis_cell_days` contains the ensemble. `modis_to_snpp_factor` is blank
because there is no single factor for the whole model; `baseline_modis_to_snpp_factor` retains
1.84. `method_version` identifies v2. Partial-week and zero-MODIS estimates remain blank,
matching the original export policy (evaluation still includes zero-count predictions).

The model JSON stores all coefficients, band definitions, fit dates, evaluation results and
source provenance. Comparison CSVs report the three methods by group and season. Existing
backfill data is not altered or relabelled as calibrated data.

In [9]:
def csv_payloads(periods, weekly, monthly, multiplier, seasons, digest, output_dir):
    payloads = {}
    for year, period in periods.items():
        rows = pd.concat([weekly[year], monthly[year]], ignore_index=True)
        estimate_ok = ~rows["is_partial_week"] & rows[MODIS].gt(0)
        rows["baseline_calibrated_modis_cell_days"] = (rows[MODIS] * float(multiplier)).where(estimate_ok)
        rows[ESTIMATE] = pd.Series(predict(rows), index=rows.index).where(estimate_ok)
        rows = rows.drop(columns="medium_frp_cell_days")
        role = {EARLY_YEAR: "early_transfer_check", LATE_YEAR: "late_transfer_check"}.get(year, "calibration")
        if year in TRANSFER_YEARS:
            role = "later_transfer_check"
        rows = rows.assign(
            calibration_role=role, coverage_status="unverified", viirs_type_status="provisional",
            source_archive_sha256=digest if year in PILOT_YEARS else "",
            timezone=TIMEZONE, grid_crs=GRID_CRS, cell_size_m=CELL_SIZE_M,
            method_version=METHOD, modis_to_snpp_factor=float("nan"),
            baseline_modis_to_snpp_factor=float(multiplier),
            calibration_model_status="retrospective_experimental", **period.provenance,
        )
        # Publish the original seasonal diagnostic only in its evaluation season.
        seasonal_rows = seasons if year == LATE_YEAR else seasons.iloc[:0]
        rows = rows.merge(seasonal_rows, on=PERIOD_KEYS, how="left", validate="one_to_one")
        rows["seasonal_reference_year_count"] = rows["seasonal_reference_year_count"].astype("Int64")
        target = output_dir / folder(year) / "harmonized_activity.csv"
        payloads[target] = rows.to_csv(index=False, date_format="%Y-%m-%d", float_format="%.12g", lineterminator="\n").encode("utf-8")
    return payloads


def stage(path, data, cleanup):
    with tempfile.NamedTemporaryFile(dir=path.parent, prefix=".harmonization-", suffix=".tmp", delete=False) as stream:
        staged = Path(stream.name)
        cleanup.callback(staged.unlink, missing_ok=True)
        stream.write(data)
    return staged


def publish(payloads, output_dir):
    for path in payloads:
        path.parent.mkdir(parents=True, exist_ok=True)
    lock = output_dir / ".pantanal_harmonization.lock"
    with ExitStack() as cleanup:
        lock.touch(exist_ok=False)
        cleanup.callback(lock.unlink, missing_ok=True)
        pending = {path: stage(path, data, cleanup) for path, data in payloads.items()}
        previous = {path: stage(path, path.read_bytes(), cleanup) for path in payloads if path.exists()}
        with ExitStack() as rollback:
            for path, staged in pending.items():
                staged.replace(path)
                if path in previous:
                    rollback.callback(previous[path].replace, path)
                else:
                    rollback.callback(path.unlink)
            rollback.pop_all()


payloads = csv_payloads(periods, weekly, monthly, multiplier, seasons, digest, OUTPUT_DIR)
model_document = {
    "method_version": METHOD, "included_satellites": ["Aqua", "Terra", "SNPP"],
    "fit_seasons": list(FIT_YEARS), "training_full_weeks": training_weeks,
    "training_months": len(training_months), "baseline_factor": float(multiplier),
    "baseline_weight": 0.5, "frp_weight": 0.5,
    "frp_coefficients_low_medium_high": frp_coefficients.tolist(),
    "frp_bands_mw": ["[0,10)", "[10,50)", "[50,infinity)"],
    "frp_aggregation": "maximum qualifying Aqua/Terra FRP per 2 km grid cell and local date",
    "timezone": TIMEZONE, "grid_crs": GRID_CRS, "cell_size_m": CELL_SIZE_M,
    "study_bounds_wgs84": BOUNDS, "pilot_archive_sha256": digest,
    "sources": {folder(year): period.provenance for year, period in periods.items()},
    "validation": comparisons,
    "limitations": "Historical evaluation periods were inspected during model development. Retrospective cross-sensor agreement only; fresh validation required. No observation masks. All-month scores include fitting data.",
}
payloads[OUTPUT_DIR / "calibration_model.json"] = (json.dumps(model_document, indent=2, allow_nan=False) + "\n").encode("utf-8")
payloads[OUTPUT_DIR / "model_comparison.csv"] = comparison.to_csv(index=False).encode("utf-8")
payloads[OUTPUT_DIR / "per_season_metrics.csv"] = per_season_comparison.to_csv(index=False).encode("utf-8")
publish(payloads, OUTPUT_DIR)
print(f"Wrote {len(periods)} activity CSVs")
for path in payloads:
    print(path)

Wrote 14 activity CSVs
pantanal_harmonized_frp_ensemble\2012-2013\harmonized_activity.csv
pantanal_harmonized_frp_ensemble\2013-2014\harmonized_activity.csv
pantanal_harmonized_frp_ensemble\2014-2015\harmonized_activity.csv
pantanal_harmonized_frp_ensemble\2015-2016\harmonized_activity.csv
pantanal_harmonized_frp_ensemble\2016-2017\harmonized_activity.csv
pantanal_harmonized_frp_ensemble\2017-2018\harmonized_activity.csv
pantanal_harmonized_frp_ensemble\2018-2019\harmonized_activity.csv
pantanal_harmonized_frp_ensemble\2019-2020\harmonized_activity.csv
pantanal_harmonized_frp_ensemble\2020-2021\harmonized_activity.csv
pantanal_harmonized_frp_ensemble\2021-2022\harmonized_activity.csv
pantanal_harmonized_frp_ensemble\2022-2023\harmonized_activity.csv
pantanal_harmonized_frp_ensemble\2023-2024\harmonized_activity.csv
pantanal_harmonized_frp_ensemble\2024-2025\harmonized_activity.csv
pantanal_harmonized_frp_ensemble\2025-2026\harmonized_activity.csv
pantanal_harmonized_frp_ensemble\calibr

## 10. Download the output ZIP

The ZIP includes all seasonal CSVs plus the model JSON and comparison CSVs. On Kaggle,
download it from the notebook's **Output** pane after running or saving a version with outputs.
The filename contains `frp_ensemble` to distinguish it from the original 1.84-only export.

In [10]:
first_year, last_year = min(periods), max(periods) + 1
zip_path = OUTPUT_DIR / f"pantanal_harmonized_frp_ensemble_{first_year}-{last_year}.zip"

buffer = io.BytesIO()
with ZipFile(buffer, "w", compression=ZIP_DEFLATED) as archive:
    for path, content in payloads.items():
        archive.writestr(path.relative_to(OUTPUT_DIR).as_posix(), content)

with ExitStack() as cleanup:
    staged = stage(zip_path, buffer.getvalue(), cleanup)
    staged.replace(zip_path)

print(f"Packaged {len(payloads)} files in {zip_path}")

Packaged 17 files in pantanal_harmonized_frp_ensemble\pantanal_harmonized_frp_ensemble_2012-2026.zip
